# Disease Explanation Based on Symptoms

This notebook demonstrates LLM-generated explanations for diseases based on the output produced by the previous pipeline layer — **Neo4j inference**.

The Neo4j inference layer scores candidate diseases by matching a patient's reported symptoms against a knowledge graph. The results (matched symptoms, missing symptoms, scores, etc.) are passed to a language model (LLM) which then generates a structured clinical explanation: the most likely diagnosis, differential diagnoses, exclusion reasoning, and a recommendation for next steps.

## Setup

Install the required packages:
- `groq` — Python client for the Groq API, used to call the LLM
- `openai` - Python client for the OpenAI API, used to call the LLM
- `python-dotenv` — loads environment variables (e.g. API key) from a `.env` file

In [2]:
%pip install groq python-dotenv openai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Option 1 (Cloud LLM)

Load the API key from the `.env` file and initialize the Groq client. The `MODEL` constant defines which LLM will be used for all subsequent calls.

> **Note:** The API key is stored in a `.env` file under the variable name `LLM_API_KEY` to avoid hardcoding secrets in the notebook.

In [14]:
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("LLM_API_KEY")

client = Groq(api_key=groq_api_key)

MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"

## Option 2 (local LLM)

In [3]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

MODEL = "llama3:8b"

## System Prompt

The system prompt defines the LLM's role and expected behavior. It instructs the model to act as a **Clinical Decision Support Assistant** — interpreting the output of the Neo4j symptom-matching engine and generating structured clinical explanations.

The prompt specifies:
- **Input structure** — what fields to expect per disease candidate (scores, matched/missing symptoms, filter status, etc.)
- **Task** — how to rank diseases, assign confidence levels, and populate each output field
- **Reasoning rules** — clinical language to use, what to abstract away (no technical jargon like "IDF" or "Neo4j"), and how to structure the analysis into *Primary Analysis*, *Differential Comparison*, and *Exclusion Criteria*
- **Output format** — a strict JSON schema the model must follow

**Output format:**
```json
{
  "most_likely": "disease name",
  "confidence": "high | moderate | low",
  "differentials": ["disease1", "disease2"],
  "excluded_conditions": ["disease1", "disease2"],
  "reasoning": {
    "primary_analysis": "Detailed clinical paragraph about the top disease.",
    "differential_comparison": "Detailed comparison with other potential candidates.",
    "exclusion_criteria": "Detailed explanation why diseases in excluded_conditions were rejected."
  },
  "recommendation": "Next steps: specialists, lab work, and specific missing symptoms to verify."
}
```

In [15]:
from pathlib import Path

BASE_DIR = Path.cwd()
prompt_path = BASE_DIR / "tests" / "xai-llm-test" / "prompts" / "disease-explanation-prompt"

REASONING_PROMPT = prompt_path.read_text(encoding="utf-8")

## Input Formatting

The `format_reasoning_input` function converts the raw list of disease candidate dictionaries (output of the Neo4j inference layer) into a plain-text string that the LLM can easily parse.

Each disease entry is formatted as a bullet point with all relevant fields:
- `URI`, `Normalized Score`, `Filter Status`, `Match Count`, `Total Symptoms`
- `Disease Coverage`, `Input Coverage`, `Blocking Symptoms`
- `Matched List`, `Missing List`

This structured text becomes the **user message** sent to the LLM (the system prompt is sent separately).

In [16]:
def format_reasoning_input(disease_results: list[dict]) -> str:
    lines = []
    for r in disease_results:
        lines.append(
            f"- {r['disease']} (\n"
            f"  URI: {r.get('uri', 'N/A')},\n"
            f"  Normalized Score: {round(r.get('normalized_score', 0), 4)},\n"
            f"  Filter Status: {r.get("passed_filter")},\n"
            f"  Match Count: {r.get("match_count", 0)},\n"
            f"  Total Symptoms: {r.get("total_symptom_count", 0)},\n"
            f"  Disease Coverage: {r.get('disease_coverage_pct', 'N/A')}, "
            f"  Input Coverage: {r.get('input_coverage_pct', 'N/A')}, "
            f"  Blocking Symptoms: {r.get("blocking_symptoms", [])},\n"
            f"  Matched List: {r.get('matched_symptoms', [])},\n"
            f"  Missing List: {r.get('missing_symptoms', [])})\n"
        )
    return "\n".join(lines)

## Reasoning Layer

The `reasoning_layer` function is the main entry point for LLM inference. It takes the list of disease candidates, formats them via `format_reasoning_input`, and sends them to the LLM.

Key implementation details:
- **Temperature `0.1`** — keeps the output deterministic and clinically consistent (low creativity)
- **`response_format: json_object`** — forces the model to return valid JSON
- **`max_tokens: 2500`** — enough headroom for detailed clinical reasoning across all output fields
- **Retry logic** — up to 3 attempts if the response is not valid JSON; falls back to a safe default dictionary with `"N/A"` values if all attempts fail
- **Regex fallback** — if `json.loads()` fails, tries to extract a JSON object from the raw response string before giving up

In [17]:
import re
import time
import json

def reasoning_layer(disease_results: list[dict], max_retries: int = 3) -> dict:
    formatted_input = format_reasoning_input(disease_results)

    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": REASONING_PROMPT},
                    {"role": "user", "content": formatted_input}
                ],
                temperature=0.1,
                top_p=1.0,
                max_tokens=2500,
                stream=False,
                response_format={"type": "json_object"},
            )

            raw = completion.choices[0].message.content

            try:
                result = json.loads(raw)
                return result
            except json.JSONDecodeError:
                pass

            match = re.search(r'\{.*\}', raw, re.DOTALL)
            if match:
                result = json.loads(match.group())
                return result

            print(f"Attempt {attempt + 1}/{max_retries} did not return valid JSON. Trying again...")
            time.sleep(1)

        except Exception as e:
            print(f"Error on attempt {attempt + 1}/{max_retries}: {e}")
            time.sleep(1)
            
    print("All attempts failed.")
    return {
        "most_likely": "N/A",
        "confidence": "N/A",
        "differentials": [],
        "excluded_conditions": [],
        "reasoning": {
            "primary_analysis": "N/A",
            "differential_comparison": "N/A",
            "exclusion_criteria": "N/A"
        },
        "recommendation": "N/A"
    }

## Test Cases

The following cells run `reasoning_layer` against four pre-defined test cases loaded from JSON files. Each test case contains the Neo4j inference output for a different patient symptom profile. The printed output shows the LLM's structured clinical explanation for that case.

## Test case 1

In [18]:
from pathlib import Path
import json

BASE_DIR = Path.cwd()

test_cases_path = BASE_DIR / "tests" / "xai-llm-test" / "test-1.json"

with open(test_cases_path, "r", encoding="utf-8") as f:
    test_case_1 = json.load(f)

print(f"Successfully loaded data for first test")

Successfully loaded data for first test


In [19]:
result = reasoning_layer(test_case_1)

RESULTS_PATH = BASE_DIR / "tests" / "xai-llm-test" / "results" / "result-1.json"
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print(f"Results saved to: {RESULTS_PATH}")

print("Most likely:", result["most_likely"])
print("Confidence:", result["confidence"])
print("Differentials:", result["differentials"])
print("Excluded Diseases:", result["excluded_conditions"])
print("Reasoning - Primary Analysis:", result["reasoning"]["primary_analysis"])
print("Reasoning - Differential Comparison:", result["reasoning"]["differential_comparison"])
print("Reasoning - Exclusion Criteria:", result["reasoning"]["exclusion_criteria"])
print("Recommendation:", result["recommendation"])

Results saved to: c:\Users\night\OneDrive\Desktop\NeSy\notebooks\tests\xai-llm-test\results\result-1.json
Most likely: nonparalytic poliomyelitis
Confidence: high
Differentials: ['Ebola virus disease', 'poliomyelitis']
Excluded Diseases: ['Marburg hemorrhagic fever', 'West Nile fever']
Reasoning - Primary Analysis: The top-ranked disease, nonparalytic poliomyelitis, shows a strong clinical correlation with the patient's symptoms, with a high degree of specificity. The patient's symptoms, including headache, pharynx inflammation, fever, and vomiting, match well with the disease profile. The absence of blocking symptoms and a high disease coverage of 66.7% and input coverage of 80.0% further support this diagnosis. The missing symptoms, fatigue and stiff neck, are not critical for the initial assessment.
Reasoning - Differential Comparison: Ebola virus disease and poliomyelitis are considered differentials due to their similar symptom profiles. However, Ebola virus disease has a lower cl

## Test case 2

In [20]:
from pathlib import Path
import json

BASE_DIR = Path.cwd()

test_cases_path = BASE_DIR / "tests" / "xai-llm-test" / "test-2.json"

with open(test_cases_path, "r", encoding="utf-8") as f:
    test_case_2 = json.load(f)

print(f"Successfully loaded data for second test")

Successfully loaded data for second test


In [21]:
result = reasoning_layer(test_case_2)

RESULTS_PATH = BASE_DIR / "tests" / "xai-llm-test" / "results" / "result-2.json"
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print(f"Results saved to: {RESULTS_PATH}")

print("Most likely:", result["most_likely"])
print("Confidence:", result["confidence"])
print("Differentials:", result["differentials"])
print("Excluded Diseases:", result["excluded_conditions"])
print("Reasoning - Primary Analysis:", result["reasoning"]["primary_analysis"])
print("Reasoning - Differential Comparison:", result["reasoning"]["differential_comparison"])
print("Reasoning - Exclusion Criteria:", result["reasoning"]["exclusion_criteria"])
print("Recommendation:", result["recommendation"])

Results saved to: c:\Users\night\OneDrive\Desktop\NeSy\notebooks\tests\xai-llm-test\results\result-2.json
Most likely: hepatitis E
Confidence: high
Differentials: ['hepatitis B', 'hepatitis C', 'hepatitis A']
Excluded Diseases: ['hepatitis D']
Reasoning - Primary Analysis: The patient presents with a strong clinical correlation for hepatitis E, given the high normalized score of 10.476 and a Filter Status that is clinically consistent with the reported absences. The disease coverage is 45.5%, and the input coverage is 100.0%, indicating that all reported symptoms match this disease. The matched symptoms include fatigue, nausea, jaundice, acholic stool, and dark urine. Although some symptoms like abdominal pain, joint pain, and fever are missing, the overall profile suggests a high likelihood of hepatitis E. The absence of blocking symptoms further supports this consideration.
Reasoning - Differential Comparison: Hepatitis B, C, and A are considered differentials due to their similar sy

## Test case 3

In [ ]:
from pathlib import Path
import json

BASE_DIR = Path.cwd()

test_cases_path = BASE_DIR / "tests" / "xai-llm-test" / "test-3.json"

with open(test_cases_path, "r", encoding="utf-8") as f:
    test_case_3 = json.load(f)

print(f"Successfully loaded data for third test")

Uspešno učitani podaci za treći test


In [93]:
result = reasoning_layer(test_case_3)

RESULTS_PATH = BASE_DIR / "tests" / "xai-llm-test" / "results" / "result-3.json"
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print(f"Results saved to: {RESULTS_PATH}")

print("Most likely:", result["most_likely"])
print("Confidence:", result["confidence"])
print("Differentials:", result["differentials"])
print("Excluded Diseases:", result["excluded_conditions"])
print("Reasoning - Primary Analysis:", result["reasoning"]["primary_analysis"])
print("Reasoning - Differential Comparison:", result["reasoning"]["differential_comparison"])
print("Reasoning - Exclusion Criteria:", result["reasoning"]["exclusion_criteria"])
print("Recommendation:", result["recommendation"])

Results saved to: c:\Users\night\OneDrive\Desktop\NeSy\notebooks\tests\xai-llm-test\results\result-3.json
Most likely: West Nile encephalitis
Confidence: low
Differentials: ['Powassan encephalitis', 'Eastern equine encephalitis']
Excluded Diseases: ['Japanese encephalitis', 'St. Louis encephalitis']
Reasoning - Primary Analysis: West Nile encephalitis shows a strong clinical correlation with the patient's presentation, matching headache, confusion, coma, stiff neck, and high fever. All reported symptoms are accounted for, and no mandatory symptoms were denied, making the disease consistent with the reported absences. The missing list (paralysis, tremor, disorientation, convulsion, muscle weakness, paresthesia, stupor) reflects features that were not observed but are part of the full disease spectrum, reducing overall disease coverage to 41.7%. Nevertheless, the specificity of the matched triad of fever, meningeal irritation, and altered consciousness supports West Nile as the leading h

## Test case 4

In [ ]:
from pathlib import Path
import json

BASE_DIR = Path.cwd()

test_cases_path = BASE_DIR / "tests" / "xai-llm-test" / "test-4.json"

with open(test_cases_path, "r", encoding="utf-8") as f:
    test_case_4 = json.load(f)

print(f"Successfully loaded data for fourth test")

Uspešno učitani podaci za četvrti test


In [94]:
result = reasoning_layer(test_case_4)

RESULTS_PATH = BASE_DIR / "tests" / "xai-llm-test" / "results" / "result-4.json"
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print(f"Results saved to: {RESULTS_PATH}")

print("Most likely:", result["most_likely"])
print("Confidence:", result["confidence"])
print("Differentials:", result["differentials"])
print("Excluded Diseases:", result["excluded_conditions"])
print("Reasoning - Primary Analysis:", result["reasoning"]["primary_analysis"])
print("Reasoning - Differential Comparison:", result["reasoning"]["differential_comparison"])
print("Reasoning - Exclusion Criteria:", result["reasoning"]["exclusion_criteria"])
print("Recommendation:", result["recommendation"])

Results saved to: c:\Users\night\OneDrive\Desktop\NeSy\notebooks\tests\xai-llm-test\results\result-4.json
Most likely: Powassan encephalitis
Confidence: moderate
Differentials: ['nonparalytic poliomyelitis', 'poliomyelitis', 'La Crosse encephalitis']
Excluded Diseases: []
Reasoning - Primary Analysis: Powassan encephalitis shows a strong clinical correlation with the patient's presentation, matching all five reported symptoms: headache, fever, vomiting, seizure, and stiff neck. The disease profile also lists several additional neurologic signs (tremor, disorientation, convulsion, spastic paralysis, stupor) that were not reported, accounting for the moderate disease coverage. No mandatory symptoms were reported as absent, so the condition remains clinically consistent with the reported absences. The high specificity of the matched symptoms, particularly the combination of fever, stiff neck, and seizure, drives the elevated score. Overall, the pattern aligns well with an arboviral enceph